In [ ]:
try:
    import json, os, subprocess, sys, time, socket, threading, traceback
    R = {"started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
         "python": sys.version.split()[0], "euid": os.geteuid(), "steps": []}
    def step(name, fn):
        t0=time.time()
        try:
            out=fn(); R["steps"].append({"step":name,"ok":True,"seconds":round(time.time()-t0,1),"detail":out}); return out
        except Exception as e:
            R["steps"].append({"step":name,"ok":False,"seconds":round(time.time()-t0,1),
                               "error":f"{type(e).__name__}: {e}"[:900],
                               "trace":traceback.format_exc()[-900:]}); raise
    def sh(cmd, timeout=1800, env=None):
        r=subprocess.run(cmd,capture_output=True,text=True,timeout=timeout,env=env)
        if r.returncode: raise RuntimeError(f"rc={r.returncode}: {(r.stderr or r.stdout)[-700:]}")
        return (r.stdout or "")[-200:]
    PWENV = dict(os.environ); PWENV["PLAYWRIGHT_BROWSERS_PATH"]="/tmp/pw"
    step("pip_install", lambda: sh([sys.executable,"-m","pip","install","playwright==1.47.0","pillow"]))
    # install-deps knows the right package names for this distro (noble renamed
    # several to *t64), which is why we do not hand-list them.
    step("install_system_deps", lambda: sh([sys.executable,"-m","playwright","install-deps","chromium"]))
    step("install_chromium", lambda: sh([sys.executable,"-m","playwright","install","chromium"], env=PWENV))
    PAGE=b"""<!doctype html><html><head><meta charset=utf-8><style>
    body{margin:0;background:#0b1b2b;color:#fff;font-family:sans-serif}.w{padding:40px}
    h1{font-size:40px}.g{display:flex;gap:14px;margin-top:24px}.b{flex:1;height:160px;
    border-radius:10px;display:flex;align-items:center;justify-content:center;font-size:26px;
    font-weight:700;color:#08111c}.a{background:#37d4a7}.c{background:#ffc857}.d{background:#ff6b6b}
    .e{background:#7cc6ff}</style></head><body><div class=w>
    <h1>SPCS container-runtime browser smoke test</h1><p>Real Chromium, real localhost page.</p>
    <div class=g><div class="b a">ALPHA</div><div class="b c">BRAVO</div>
    <div class="b d">CHARLIE</div><div class="b e">DELTA</div></div></div></body></html>"""
    def serve():
        import http.server
        with socket.socket() as s: s.bind(("127.0.0.1",0)); port=s.getsockname()[1]
        class H(http.server.BaseHTTPRequestHandler):
            def do_GET(self):
                self.send_response(200); self.send_header("Content-Type","text/html")
                self.send_header("Content-Length",str(len(PAGE))); self.end_headers(); self.wfile.write(PAGE)
            def log_message(self,*a): pass
        srv=http.server.HTTPServer(("127.0.0.1",port),H)
        threading.Thread(target=srv.serve_forever,daemon=True).start(); return port
    port = step("start_local_server", serve)
    def capture():
        os.environ["PLAYWRIGHT_BROWSERS_PATH"]="/tmp/pw"
        from playwright.sync_api import sync_playwright
        os.makedirs("/tmp/smoke",exist_ok=True)
        with sync_playwright() as p:
            b=p.chromium.launch(args=["--no-sandbox","--disable-dev-shm-usage"])
            ver=b.version
            pg=b.new_page(viewport={"width":1600,"height":900},device_scale_factor=2)
            pg.goto(f"http://127.0.0.1:{port}/",wait_until="networkidle",timeout=30000)
            pg.wait_for_timeout(500)
            pg.screenshot(path="/tmp/smoke/smoke.png",full_page=True)
            b.close()
        return {"chromium_version":ver,"png_bytes":os.path.getsize("/tmp/smoke/smoke.png")}
    step("capture_screenshot", capture)
    def blank():
        from PIL import Image
        with Image.open("/tmp/smoke/smoke.png") as im:
            im=im.convert("RGB"); w,h=im.size; cols=im.getcolors(maxcolors=1000000)
        if cols is None: return {"width":w,"height":h,"distinct_colours":">1e6","non_blank":True}
        tot=sum(c for c,_ in cols); dom=max(c for c,_ in cols); share=dom/tot
        d={"width":w,"height":h,"distinct_colours":len(cols),"dominant_share":round(share,4),
           "non_blank": bool(len(cols)>=8 and share<0.98)}
        if not d["non_blank"]: raise RuntimeError(f"blank/near-uniform screenshot: {d}")
        return d
    step("blank_check", blank)
    R["ok"]=all(s["ok"] for s in R["steps"])
except Exception:
    R['ok']=False
from snowflake.snowpark.context import get_active_session
s=get_active_session()
R["finished_utc"]=time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime())
with open("/tmp/smoke_result.json","w") as fh: json.dump(R,fh,indent=1)
s.file.put("/tmp/smoke_result.json","@WBR_DB.DBX.VALIDATION_CAPTURE_STAGE/smoke-test",
           auto_compress=False,overwrite=True)
if os.path.exists("/tmp/smoke/smoke.png"):
    s.file.put("/tmp/smoke/smoke.png","@WBR_DB.DBX.VALIDATION_CAPTURE_STAGE/smoke-test",
               auto_compress=False,overwrite=True)
print(json.dumps(R,indent=1)[:2500])
